<img src="https://github.com/hernancontigiani/ceia_memorias_especializacion/raw/master/Figures/logoFIUBA.jpg" width="500" align="center">


# Procesamiento de lenguaje natural
## Modelo de lenguaje con tokenización por caracteres

### Consigna
- Seleccionar un corpus de texto sobre el cual entrenar el modelo de lenguaje.
- Realizar el pre-procesamiento adecuado para tokenizar el corpus, estructurar el dataset y separar entre datos de entrenamiento y validación.
- Proponer arquitecturas de redes neuronales basadas en unidades recurrentes para implementar un modelo de lenguaje.
- Con el o los modelos que consideren adecuados, generar nuevas secuencias a partir de secuencias de contexto con las estrategias de greedy search y beam search determístico y estocástico. En este último caso observar el efecto de la temperatura en la generación de secuencias.


### Sugerencias
- Durante el entrenamiento, guiarse por el descenso de la perplejidad en los datos de validación para finalizar el entrenamiento. Para ello se provee un callback.
- Explorar utilizar SimpleRNN (celda de Elman), LSTM y GRU.
- rmsprop es el optimizador recomendado para la buena convergencia. No obstante se pueden explorar otros.


In [1]:
import random
import io
import pickle

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

from tensorflow import keras
from tensorflow.keras import layers
from keras.utils import to_categorical
from keras.models import Sequential
from keras.layers import Dense, LSTM, Embedding, Dropout
from tensorflow.keras.losses import SparseCategoricalCrossentropy

### Datos

In [2]:
# descargar de textos.info
import urllib.request

# Para leer y parsear el texto en HTML de wikipedia
import bs4 as bs

In [ ]:
raw_html = urllib.request.urlopen('https://www.textos.info/julio-verne/la-vuelta-al-mundo-en-80-dias/ebook')
raw_html = raw_html.read()

# Parsear artículo, 'lxml' es el parser a utilizar
article_html = bs.BeautifulSoup(raw_html, 'lxml')

# Encontrar todos los párrafos del HTML (bajo el tag <p>)
# y tenerlos disponible como lista
article_paragraphs = article_html.find_all('p')

article_text = ''

for para in article_paragraphs:
    article_text += para.text + ' '

# pasar todo el texto a minúscula
article_text = article_text.lower()

In [ ]:
# en article text se encuentra el texto de todo el libro
article_text[:1000]

### Elegir el tamaño del contexto

En este caso, como el modelo de lenguaje es por caracteres, todo un gran corpus
de texto puede ser considerado un documento en sí mismo y el tamaño de contexto
puede ser elegido con más libertad en comparación a un modelo de lenguaje tokenizado por palabras y dividido en documentos más acotados.

In [8]:
# seleccionamos el tamaño de contexto
max_context_size = 100

In [3]:
# Usaremos las utilidades de procesamiento de textos y secuencias de Keras
from tensorflow.keras.utils import pad_sequences # se utilizará para padding

In [ ]:
# en este caso el vocabulario es el conjunto único de caracteres que existe en todo el texto
chars_vocab = set(article_text)

In [ ]:
# la longitud de vocabulario de caracteres es:
len(chars_vocab)

67

In [ ]:
# Construimos los dicionarios que asignan índices a caracteres y viceversa.
# El diccionario `char2idx` servirá como tokenizador.
char2idx = {k: v for v,k in enumerate(chars_vocab)}
idx2char = {v: k for k,v in char2idx.items()}

###  Tokenizar

In [ ]:
# tokenizamos el texto completo
tokenized_text = [char2idx[ch] for ch in article_text]

In [ ]:
tokenized_text[:1000]

### Organizando y estructurando el dataset

In [ ]:
# separaremos el dataset entre entrenamiento y validación.
# `p_val` será la proporción del corpus que se reservará para validación
# `num_val` es la cantidad de secuencias de tamaño `max_context_size` que se usará en validación
p_val = 0.1
num_val = int(np.ceil(len(tokenized_text)*p_val/max_context_size))

In [ ]:
# separamos la porción de texto utilizada en entrenamiento de la de validación.
train_text = tokenized_text[:-num_val*max_context_size]
val_text = tokenized_text[-num_val*max_context_size:]

In [ ]:
tokenized_sentences_val = [val_text[init*max_context_size:init*(max_context_size+1)] for init in range(num_val)]

In [ ]:
tokenized_sentences_train = [train_text[init:init+max_context_size] for init in range(len(train_text)-max_context_size+1)]

In [ ]:
X = np.array(tokenized_sentences_train[:-1])
y = np.array(tokenized_sentences_train[1:])

Nótese que estamos estructurando el problema de aprendizaje como *many-to-many*:

Entrada: secuencia de tokens [$x_0$, $x_1$, ..., $x_N$]

Target: secuencia de tokens [$x_1$, $x_2$, ..., $x_{N+1}$]

De manera que la red tiene que aprender que su salida deben ser los tokens desplazados en una posición y un nuevo token predicho (el N+1).

La ventaja de estructurar el aprendizaje de esta manera es que para cada token de target se propaga una señal de gradiente por el grafo de cómputo recurrente, que es mejor que estructurar el problema como *many-to-one* en donde sólo una señal de gradiente se propaga.

En este punto tenemos en la variable `tokenized_sentences` los versos tokenizados. Vamos a quedarnos con un conjunto de validación que utilizaremos para medir la calidad de la generación de secuencias con la métrica de Perplejidad.

In [ ]:
X.shape

In [ ]:
X[0,:10]

In [ ]:
y[0,:10]

In [ ]:
vocab_size = len(chars_vocab)

# Definiendo el modelo

In [4]:
from keras.layers import Input, TimeDistributed, CategoryEncoding, SimpleRNN, Dense,LSTM, GRU
from keras.models import Model, Sequential

El modelo que se propone como ejemplo consume los índices de los tokens y los transforma en vectores OHE (en este caso no entrenamos una capa de embedding para caracteres). Esa transformación se logra combinando las capas `CategoryEncoding` que transforma a índices a vectores OHE y `TimeDistributed` que aplica la capa a lo largo de la dimensión "temporal" de la secuencia.

In [ ]:
model = Sequential()

model.add(TimeDistributed(CategoryEncoding(num_tokens=vocab_size, output_mode = "one_hot"),input_shape=(None,1)))
model.add(SimpleRNN(200, return_sequences=True, dropout=0.1, recurrent_dropout=0.1 ))
model.add(Dense(vocab_size, activation='softmax'))
model.compile(loss='sparse_categorical_crossentropy', optimizer='rmsprop')

model.summary()


### Definir el modelo

Dado que por el momento no hay implementaciones adecuadas de la perplejidad que puedan operar en tiempo de entrenamiento, armaremos un Callback *ad-hoc* que la calcule en cada epoch.

**Nota**: un Callback es una rutina gatillada por algún evento, son muy útiles para relevar datos en diferentes momentos del desarrollo del modelo. En este caso queremos hacer un cálculo cada vez que termina una epoch de entrenamiento.

In [5]:
class PplCallback(keras.callbacks.Callback):

    '''
    Este callback es una solución ad-hoc para calcular al final de cada epoch de
    entrenamiento la métrica de Perplejidad sobre un conjunto de datos de validación.
    La perplejidad es una métrica cuantitativa para evaluar la calidad de la generación de secuencias.
    Además implementa la finalización del entrenamiento (Early Stopping)
    si la perplejidad no mejora después de `patience` epochs.
    '''

    def __init__(self, val_data, history_ppl,patience=5):
      # El callback lo inicializamos con secuencias de validación sobre las cuales
      # mediremos la perplejidad
      self.val_data = val_data

      self.target = []
      self.padded = []

      count = 0
      self.info = []
      self.min_score = np.inf
      self.patience_counter = 0
      self.patience = patience

      # nos movemos en todas las secuencias de los datos de validación
      for seq in self.val_data:

        len_seq = len(seq)
        # armamos todas las subsecuencias
        subseq = [seq[:i] for i in range(1,len_seq)]
        self.target.extend([seq[i] for i in range(1,len_seq)])

        if len(subseq)!=0:

          self.padded.append(pad_sequences(subseq, maxlen=max_context_size, padding='pre'))

          self.info.append((count,count+len_seq))
          count += len_seq

      self.padded = np.vstack(self.padded)


    def on_epoch_end(self, epoch, logs=None):

        # en `scores` iremos guardando la perplejidad de cada secuencia
        scores = []

        predictions = self.model.predict(self.padded,verbose=0)

        # para cada secuencia de validación
        for start,end in self.info:

          # en `probs` iremos guardando las probabilidades de los términos target
          probs = [predictions[idx_seq,-1,idx_vocab] for idx_seq, idx_vocab in zip(range(start,end),self.target[start:end])]

          # calculamos la perplejidad por medio de logaritmos
          scores.append(np.exp(-np.sum(np.log(probs))/(end-start)))

        # promediamos todos los scores e imprimimos el valor promedio
        current_score = np.mean(scores)
        history_ppl.append(current_score)
        print(f'\n mean perplexity: {current_score} \n')

        # chequeamos si tenemos que detener el entrenamiento
        if current_score < self.min_score:
          self.min_score = current_score
          self.model.save("my_model.keras")
          print("Saved new model!")
          self.patience_counter = 0
        else:
          self.patience_counter += 1
          if self.patience_counter == self.patience:
            print("Stopping training...")
            self.model.stop_training = True


### Entrenamiento

In [ ]:
# fiteamos, nótese el agregado del callback con su inicialización. El batch_size lo podemos seleccionar a mano
# en general, lo mejor es escoger el batch más grande posible que minimice el tiempo de cada época.
# En la variable `history_ppl` se guardarán los valores de perplejidad para cada época.
history_ppl = []
hist = model.fit(X, y, epochs=20, callbacks=[PplCallback(tokenized_sentences_val,history_ppl)], batch_size=256)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Entrenamiento
epoch_count = range(1, len(history_ppl) + 1)
sns.lineplot(x=epoch_count,  y=history_ppl)
plt.show()

In [ ]:
# Cargamos el mejor modelo guardado del entrenamiento para hacer inferencia
model = keras.models.load_model('my_model.keras')


### Predicción del próximo caracter

In [ ]:
# Se puede usar gradio para probar el modelo
# Gradio es una herramienta muy útil para crear interfaces para ensayar modelos
# https://gradio.app/

!pip install -q gradio

In [ ]:
import gradio as gr

def model_response(human_text):

    # Encodeamos
    encoded = [char2idx[ch] for ch in human_text.lower() ]
    # Si tienen distinto largo
    encoded = pad_sequences([encoded], maxlen=max_context_size, padding='pre')

    # Predicción softmax
    y_hat = np.argmax(model.predict(encoded)[0,-1,:])


    # Debemos buscar en el vocabulario el caracter
    # que corresopnde al indice (y_hat) predicho por le modelo
    out_word = ''
    out_word = idx2char[y_hat]

    # Agrego la palabra a la frase predicha
    return human_text + out_word

iface = gr.Interface(
    fn=model_response,
    inputs=["textbox"],
    outputs="text")

iface.launch(debug=True)

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://603fc66c6e8b48d9a2.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://603fc66c6e8b48d9a2.gradio.live


### Generación de secuencias

In [6]:
def generate_seq(model, seed_text, max_length, n_words):
    """
        Exec model sequence prediction

        Args:
            model (keras): modelo entrenado
            seed_text (string): texto de entrada (input_seq)
            max_length (int): máxima longitud de la sequencia de entrada
            n_words (int): números de caracteres a agregar a la sequencia de entrada
        returns:
            output_text (string): sentencia con las "n_words" agregadas
    """
    output_text = seed_text
	# generate a fixed number of words
    for _ in range(n_words):
		# Encodeamos
        encoded = [char2idx[ch] for ch in output_text.lower() ]
		# Si tienen distinto largo
        encoded = pad_sequences([encoded], maxlen=max_length, padding='pre')

		# Predicción softmax
        y_hat = np.argmax(model.predict(encoded,verbose=0)[0,-1,:])
		# Vamos concatenando las predicciones
        out_word = ''

        out_word = idx2char[y_hat]

		# Agrego las palabras a la frase predicha
        output_text += out_word
    return output_text

In [ ]:
input_text='habia una vez'

generate_seq(model, input_text, max_length=max_context_size, n_words=30)

###  Beam search y muestreo aleatorio

In [9]:
# funcionalidades para hacer encoding y decoding

def encode(text,max_length=max_context_size):

    encoded = [char2idx[ch] for ch in text]
    encoded = pad_sequences([encoded], maxlen=max_length, padding='pre')

    return encoded

def decode(seq):
    return ''.join([idx2char[ch] for ch in seq])

In [10]:
from scipy.special import softmax

# función que selecciona candidatos para el beam search
def select_candidates(pred,num_beams,vocab_size,history_probs,history_tokens,temp,mode):

  # colectar todas las probabilidades para la siguiente búsqueda
  pred_large = []

  for idx,pp in enumerate(pred):
    pred_large.extend(np.log(pp+1E-10)+history_probs[idx])

  pred_large = np.array(pred_large)

  # criterio de selección
  if mode == 'det':
    idx_select = np.argsort(pred_large)[::-1][:num_beams] # beam search determinista
  elif mode == 'sto':
    idx_select = np.random.choice(np.arange(pred_large.shape[0]), num_beams, p=softmax(pred_large/temp)) # beam search con muestreo aleatorio
  else:
    raise ValueError(f'Wrong selection mode. {mode} was given. det and sto are supported.')

  # traducir a índices de token en el vocabulario
  new_history_tokens = np.concatenate((np.array(history_tokens)[idx_select//vocab_size],
                        np.array([idx_select%vocab_size]).T),
                      axis=1)

  # devolver el producto de las probabilidades (log) y la secuencia de tokens seleccionados
  return pred_large[idx_select.astype(int)], new_history_tokens.astype(int)


def beam_search(model,num_beams,num_words,input,temp=1,mode='det'):

    # first iteration

    # encode
    encoded = encode(input)

    # first prediction
    y_hat = model.predict(encoded,verbose=0)[0,-1,:]

    # get vocabulary size
    vocab_size = y_hat.shape[0]

    # initialize history
    history_probs = [0]*num_beams
    history_tokens = [encoded[0]]*num_beams

    # select num_beams candidates
    history_probs, history_tokens = select_candidates([y_hat],
                                        num_beams,
                                        vocab_size,
                                        history_probs,
                                        history_tokens,
                                        temp,
                                        mode)

    # beam search loop
    for i in range(num_words-1):

      preds = []

      for hist in history_tokens:

        # actualizar secuencia de tokens
        input_update = np.array([hist[i+1:]]).copy()

        # predicción
        y_hat = model.predict(input_update,verbose=0)[0,-1,:]

        preds.append(y_hat)

      history_probs, history_tokens = select_candidates(preds,
                                                        num_beams,
                                                        vocab_size,
                                                        history_probs,
                                                        history_tokens,
                                                        temp,
                                                        mode)

    return history_tokens[:,-(len(input)+num_words):]

In [ ]:
# predicción con beam search
salidas = beam_search(model,num_beams=10,num_words=20,input="habia una vez")

In [ ]:
salidas[0]

In [ ]:
# veamos las salidas
decode(salidas[0])

# Solución consigna 3

In [11]:
from google.colab import drive
import os

drive.mount('/content/drive')
#creamos los directorios donde vamos a almacenar los resultados de los modelos
BASE_DIR = '/content/drive/MyDrive/Colab/PLN'
DIR_MODELOS = os.path.join(BASE_DIR, 'modelos')
DIR_RESULTADOS = os.path.join(BASE_DIR, 'resultados')
DIR_HISTORIAL = os.path.join(BASE_DIR, 'historial')

for directorio in [DIR_MODELOS, DIR_RESULTADOS, DIR_HISTORIAL]:
    os.makedirs(directorio, exist_ok=True)

print("Directorio de resultados:", BASE_DIR)

Mounted at /content/drive
Directorio de resultados: /content/drive/MyDrive/Colab/PLN


In [12]:
# Definir un corpus
# Para ello se tomó como elección el libro Orgullo y prejuicio
# Descargado de textos.info
# max_context_size=100
text_html = urllib.request.urlopen('https://www.textos.info/jane-austen/orgullo-y-prejuicio/ebook')
text_html = text_html.read()

# Parsear artículo, 'lxml' es el parser a utilizar
article_html = bs.BeautifulSoup(text_html, 'lxml')

# Encontrar todos los párrafos del HTML (bajo el tag <p>)
# y tenerlos disponible como lista
article_paragraphs = article_html.find_all('p')
article_text = ''

for para in article_paragraphs:
    article_text += para.text + ' '

# pasar todo el texto a minúscula
article_text = article_text.lower()
#print (article_text)

# Construimos los dicionarios que asignan índices a caracteres y viceversa.
# El diccionario `char2idx` servirá como tokenizador.
#chars_vocab = set(article_text)
#len(chars_vocab)
#char2idx = {k: v for v,k in enumerate(chars_vocab)}
#idx2char = {v: k for k,v in char2idx.items()}

chars_vocab = sorted(set(article_text))

vocab_size = len(chars_vocab)

char2idx = {k: v for v, k in enumerate(chars_vocab)}
idx2char = {v: k for k, v in char2idx.items()}

print("Vocabulario:", vocab_size)



Vocabulario: 65


Realizar el pre-procesamiento adecuado para tokenizar el corpus, estructurar el dataset y separar entre datos de entrenamiento y validación.

In [14]:
#Comenzamos a tokenizar
tokenized_text = [char2idx[ch] for ch in article_text]

#Entrenamos y validamos el dataset previamente estructurado

In [15]:

import numpy as np

# Configuración experimental
PORCENTAJE_CORPUS = 0.20
max_context_size = 50
p_val = 0.10

# Tomar el primer 10 % del texto, respetando su orden
n = int(len(tokenized_text) * PORCENTAJE_CORPUS)
corpus_reducido = tokenized_text[:n]

# Separar entrenamiento y validación
num_val = max(1, int(np.ceil(
    len(corpus_reducido) * p_val / max_context_size
)))

train_text = np.asarray(
    corpus_reducido[:-num_val * max_context_size],
    dtype=np.int32
)

val_text = np.asarray(
    corpus_reducido[-num_val * max_context_size:],
    dtype=np.int32
)

NUM_VAL_CALLBACK = min(20, num_val)

tokenized_sentences_val = [
    val_text[i * max_context_size:(i + 1) * max_context_size]
    for i in range(NUM_VAL_CALLBACK)
]

print("Caracteres de entrenamiento:", len(train_text))
print("Caracteres de validación:", len(val_text))
print("Secuencias para perplexity:", len(tokenized_sentences_val))

Caracteres de entrenamiento: 120066
Caracteres de validación: 13350
Secuencias para perplexity: 20


In [25]:


class GeneradorSecuencias(tf.keras.utils.Sequence):
    def __init__(self, texto, context_size, batch_size=16,
                 shuffle=True, **kwargs):
        super().__init__(**kwargs)
        self.texto = np.asarray(texto, dtype=np.int32)
        self.context_size = context_size
        self.batch_size = batch_size
        self.shuffle = shuffle
        self.indices = np.arange(len(self.texto) - context_size)
        self.on_epoch_end()

    def __len__(self):
        return int(np.ceil(len(self.indices) / self.batch_size))

    def __getitem__(self, idx):
        inicio = idx * self.batch_size
        indices = self.indices[inicio:inicio + self.batch_size]

        X = np.stack([
            self.texto[i:i + self.context_size]
            for i in indices
        ])
        y = np.stack([
            self.texto[i + 1:i + self.context_size + 1]
            for i in indices
        ])

        return X, y

    def on_epoch_end(self):
        if self.shuffle:
            np.random.shuffle(self.indices)


train_gen = GeneradorSecuencias(
    train_text,
    context_size=max_context_size,
    batch_size=16
)

print("Lotes por época:", len(train_gen))

Lotes por época: 7501


In [26]:
arquitecturas = {
    "SimpleRNN": SimpleRNN,
    "LSTM": LSTM,
    "GRU": GRU,
}

def crear_modelo(nombre):
    model = Sequential(name=nombre)

    model.add(Input(shape=(max_context_size,)))
    model.add(Embedding(
        input_dim=vocab_size,
        output_dim=32
    ))

    model.add(
        arquitecturas[nombre](
            64,
            return_sequences=True,
            dropout=0.1,
            recurrent_dropout=0.0
        )
    )

    model.add(Dense(vocab_size, activation="softmax"))

    model.compile(
        loss="sparse_categorical_crossentropy",
        optimizer=tf.keras.optimizers.RMSprop(learning_rate=0.001)
    )

    return model

In [27]:
# Comenzamos a definir el modelo, almacenando cada corrida y sus respectivos resultados #

import os
import json
import tensorflow as tf

# Directorio principal en Google Drive
BASE_DIR = '/content/drive/MyDrive/PLN'
DIR_MODELOS = os.path.join(BASE_DIR, 'modelos')
DIR_HISTORIAL = os.path.join(BASE_DIR, 'historial')

os.makedirs(DIR_MODELOS, exist_ok=True)
os.makedirs(DIR_HISTORIAL, exist_ok=True)

resultados = {}

for nombre in arquitecturas:
    print(f"\nEntrenando {nombre}")

    dir_modelo = os.path.join(DIR_MODELOS, nombre)
    os.makedirs(dir_modelo, exist_ok=True)

    m = crear_modelo(nombre)
    print("Parámetros:", m.count_params())

    history_ppl = []
    callback = PplCallback(tokenized_sentences_val, history_ppl)

    hist = m.fit(
        train_gen,
        epochs=20,
        callbacks=[callback]
    )

    # Guardar el estado final
    m.save(os.path.join(dir_modelo, "modelo_final.keras"))

    # Guardar las métricas
    datos = {
        "arquitectura": nombre,
        "parametros": int(m.count_params()),
        "porcentaje_corpus": PORCENTAJE_CORPUS,
        "contexto": max_context_size,
        "unidades": 64,
        "loss": [float(x) for x in hist.history["loss"]],
        "perplexity": [float(x) for x in history_ppl],
        "mejor_perplexity": (
            float(min(history_ppl)) if history_ppl else None
        )
    }

    ruta_json = os.path.join(
        DIR_HISTORIAL, f"{nombre}.json"
    )

    with open(ruta_json, "w", encoding="utf-8") as f:
        json.dump(datos, f, indent=4, ensure_ascii=False)

    resultados[nombre] = datos

    with open(
        os.path.join(BASE_DIR, "resumen.json"),
        "w",
        encoding="utf-8"
    ) as f:
        json.dump(resultados, f, indent=4, ensure_ascii=False)

    print("Mejor perplexity:", datos["mejor_perplexity"])
    print("Resultados guardados en:", ruta_json)

print("\nExperimentos terminados.")



Entrenando SimpleRNN
Parámetros: 12513
Epoch 1/20
7492/7501 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 2.1042
 mean perplexity: 5.799315929412842 

Saved new model!
7501/7501 ━━━━━━━━━━━━━━━━━━━━ 47s 6ms/step - loss: 1.8996
Epoch 2/20
7498/7501 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 1.7153
 mean perplexity: 5.420916557312012 

Saved new model!
7501/7501 ━━━━━━━━━━━━━━━━━━━━ 40s 5ms/step - loss: 1.6982
Epoch 3/20
7498/7501 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 1.6625
 mean perplexity: 5.2718892097473145 

Saved new model!
7501/7501 ━━━━━━━━━━━━━━━━━━━━ 40s 5ms/step - loss: 1.6553
Epoch 4/20
7496/7501 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 1.6394
 mean perplexity: 5.237249851226807 

Saved new model!
7501/7501 ━━━━━━━━━━━━━━━━━━━━ 40s 5ms/step - loss: 1.6352
Epoch 5/20
7494/7501 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 1.6253
 mean perplexity: 5.266395092010498 

7501/7501 ━━━━━━━━━━━━━━━━━━━━ 41s 5ms/step - loss: 1.6227
Epoch 6/20
7494/7501 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss:

In [20]:
import tensorflow as tf
modelos = {}
BASE_DIR = '/content/drive/MyDrive/PLN'
DIR_MODELOS = os.path.join(BASE_DIR, 'modelos')
DIR_HISTORIAL = os.path.join(BASE_DIR, 'historial')

for nombre in ["SimpleRNN", "LSTM", "GRU"]:
    dir_modelo = os.path.join(DIR_MODELOS, nombre)
    ruta = os.path.join(dir_modelo, "modelo_final.keras")

    if os.path.exists(ruta):
        modelos[nombre] = tf.keras.models.load_model(ruta)
        print(f"{nombre} cargado correctamente")
    else:
        print(f"Archivo no encontrado: {ruta}")

print("Disponibles:", list(modelos.keys()))

SimpleRNN cargado correctamente
LSTM cargado correctamente
GRU cargado correctamente
Disponibles: ['SimpleRNN', 'LSTM', 'GRU']


#Resultados

In [23]:
modelo = modelos["GRU"]

contexto = "es una verdad universalmente reconocida"

num_beams = 3
num_chars = 200

# Beam search determinístico
resultado_det = beam_search(
    modelo,
    num_beams=num_beams,
    num_words=num_chars,
    input=contexto,
    temp=1.0,
    mode="det"
)

print("Beam search determinístico:")
print(decode(resultado_det[0]))


# Se prueba con el estocastico

for temperatura in [0.5, 1.0, 1.5]:
    resultado = beam_search(
        modelo,
        num_beams=3,
        num_words=200,
        input=contexto,
        temp=temperatura,
        mode="sto"
    )

    print(f"\nTemperatura: {temperatura}")
    print(decode(resultado[0]))

Beam search determinístico:
es una verdad universalmente reconocidad de la señorita bingley estaba de la señorita bingley estaba con el señor bingley estaba con el señor bingley estaba con el señor bingley estaba con el señor bingley estaba con el señor bingley estab

Temperatura: 0.5
es una verdad universalmente reconocidad de la señorita bingley se había de la señorita bingley se decirlo de su hermanas de su carácter de la señorita bingley se distinado con la señorita bingley por lo que se descucharía a la señorita bi

Temperatura: 1.0
es una verdad universalmente reconocidado de las 
hermanas, y lo había visitarla de compromendo a sólo la señorita bingley pero para la comportamiento y que se casa. elizabeth para que la señora 
bingley —dijo el señor bingley estaba carta

Temperatura: 1.5
es una verdad universalmente reconocidado no qué señor bennet, con el señor bingley; 
así, ¿qué y no ventaja, de las medio que se cuendad. —es con la madre—. —sería y poco de esta
 hora, ser todas l


# Análisis del trabajo

## 1. Conclusiones generales

En este trabajo se implementó y evaluó un modelo de lenguaje, utilizando como corpus la novela *Orgullo y prejuicio*, en su versión en español. El objetivo fue estudiar la capacidad de diferentes arquitecturas recurrentes para aprender patrones lingüísticos a nivel de caracteres y generar nuevas secuencias de texto a partir de un contexto inicial.

Para ello, se realizó el preprocesamiento del corpus un 20% debido a la capacidad de consumo que requeriía (las pruebas se ejecutaron con google colab), la tokenización mediante un vocabulario de caracteres, la construcción de secuencias de entrenamiento y la separación de los datos de entrenamiento y validación. Posteriormente, se implementaron y compararon tres arquitecturas: SimpleRNN, LSTM y GRU. Finalmente, se exploraron estrategias de generación de texto mediante greedy search y beam search, tanto determinístico como estocástico.

Los resultados permitieron observar diferencias entre las arquitecturas recurrentes y analizar las limitaciones de la generación automática de texto cuando se trabaja con un corpus reducido y modelos entrenados a nivel de caracteres.

## 2. Comparación de las arquitecturas recurrentes

Durante el experimento inicial, las tres arquitecturas fueron entrenadas durante cinco épocas con un subconjunto del 10 % del corpus. Se utilizó la perplexity de validación como métrica principal para comparar su capacidad predictiva.

Los mejores valores registrados fueron los siguientes:

| Arquitectura | Parámetros | Mejor perplexity |
| ------------ | ---------: | ---------------: |
| SimpleRNN    |     12.513 |            5,07  |
| LSTM         |     31.137 |            4,78  |
| GRU          |     25.121 |        **4,63 ** |

La arquitectura GRU obtuvo el menor valor de perplexity de validación, lo que indica que, en este experimento, asignó una probabilidad más favorable a los caracteres observados en los datos de validación. Además, consiguió este resultado con menos parámetros que LSTM, por lo que presentó una relación favorable entre complejidad del modelo y desempeño predictivo.

LSTM mostró una mejora progresiva de la perplexity durante las cinco épocas evaluadas, mientras que SimpleRNN presentó fluctuaciones en la métrica de validación. Estas diferencias deben interpretarse teniendo en cuenta la duración limitada del entrenamiento y el tamaño reducido del corpus utilizado.

Por lo tanto, GRU fue la arquitectura con mejor desempeño en la prueba inicial, aunque sería necesario realizar evaluaciones adicionales para determinar si esta ventaja se mantiene al incrementar la cantidad de datos y el número de épocas.

## 3. Análisis de las estrategias de generación de texto

Las pruebas de generación permitieron observar que el desempeño predictivo del modelo no garantiza, por sí solo, la producción de texto coherente.

En la generación mediante beam search determinístico con GRU, se obtuvieron secuencias que conservaban parte del contexto inicial y reproducían palabras y nombres asociados con el corpus. Sin embargo, también se observaron errores ortográficos, construcciones lingüísticas incompletas y repeticiones frecuentes de secuencias como "estaba con el señor Bingley".

Este comportamiento evidencia una limitación importante del modelo aunque aprende asociaciones entre caracteres y patrones locales del texto, no necesariamente desarrolla una representación suficiente de la estructura sintáctica y semántica global.

El beam search determinístico selecciona las secuencias candidatas según sus puntuaciones acumuladas. Por ello, puede favorecer continuaciones repetitivas si estas reciben probabilidades elevadas del modelo. Mantener varias secuencias candidatas permite explorar más alternativas que greedy search, pero no garantiza una mejora en la calidad lingüística.

La generación estocástica introduce variabilidad en la selección de candidatos. La comparación de diferentes temperaturas permite estudiar el equilibrio entre predictibilidad y diversidad: una temperatura baja favorece las alternativas más probables, mientras que una temperatura alta permite explorar alternativas menos probables, con el riesgo de producir secuencias menos coherentes. El efecto concreto debe establecerse a partir de las salidas generadas para cada temperatura.


